In [ ]:
import streamlit as st
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, roc_curve, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, load_wine

In [ ]:
st.title("Interactive Machine Learning Explorer")

In [ ]:
# Dataset selection
dataset_option = st.sidebar.selectbox("Choose Dataset", ["Upload your own", "Iris"])

if dataset_option == "Upload your own":
    uploaded_file = st.file_uploader("Upload CSV", type=["csv"])
    if uploaded_file:
        df = pd.read_csv(uploaded_file)
        st.write("Dataset Preview:", df.head())
    else:
        st.warning("Please upload a dataset to proceed.")
        df = None
elif dataset_option == "Iris":
    iris = load_iris()
    df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
    df['target'] = iris.target
    st.write("Iris Dataset loaded.")



In [ ]:
# Proceed if dataset is loaded
if df is not None:
    # Show feature and target selection
    features = df.columns[:-1]
    target = df.columns[-1]
    
    st.sidebar.header("Model Hyperparameters")
    test_size = st.sidebar.slider("Test set size (%)", 10, 50, 20)
    model_type = st.sidebar.selectbox("Model Type", ["Logistic Regression", "Decision Tree"])

    # Hyperparameters
    if model_type == "Logistic Regression":
        C = st.sidebar.slider("C (Inverse of regularization strength)", 0.01, 10.0, 1.0)
        max_iter = st.sidebar.slider("Max Iterations", 100, 1000, 200)
    elif model_type == "Decision Tree":
        max_depth = st.sidebar.slider("Max Depth", 1, 20, 5)
        min_samples_split = st.sidebar.slider("Min Samples Split", 2, 20, 2)
         # Prepare data
    X = df[features]
    y = df[target]
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size/100, random_state=42
    )
    
    # Initialize model
    if model_type == "Logistic Regression":
        model = LogisticRegression(C=C, max_iter=max_iter, solver='lbfgs', multi_class='auto')
    elif model_type == "Decision Tree":
        model = DecisionTreeClassifier(max_depth=max_depth, min_samples_split=min_samples_split)
    
    # Train
    model.fit(X_train, y_train)
    
    # Predictions
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    
    # Performance
    accuracy = accuracy_score(y_test, y_pred)
    st.write(f"**Accuracy:** {accuracy:.2f}")
    
    # ROC Curve
    if y_proba is not None:
        fpr, tpr, _ = roc_curve(y_test, y_proba, pos_label=y.unique()[1] if y.nunique() > 1 else y.unique()[0])
        roc_auc = roc_auc_score(y_test, y_proba)
        
        fig, ax = plt.subplots()
        ax.plot(fpr, tpr, label=f'ROC curve (area = {roc_auc:.2f})')
        ax.plot([0, 1], [0, 1], 'k--')
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.set_title('ROC Curve')
        ax.legend(loc='lower right')
        st.pyplot(fig)
else:
    st.info("Please upload or select a dataset to start modeling.")